> **Reshaping is about changing the structure without changing the meaning.**

There are two fundamental shapes:
- **Wide format**
   - More columns
   - Fewer rows
   - Good for **human reading, dashboards**

- **Long (tidy) format**
   - Fewer columns
   - More rows
   - Required for **ML, plotting, groupby**

**Most ML & plotting libraries expect LONG format.**

### Base DataFrame (Wide Format)

In [8]:
import pandas as pd

df = pd.DataFrame({
    "city": ["Pune", "Mumbai", "Delhi"],
    "sales_Q1": [100, 200, 150],
    "sales_Q2": [120, 220, 180],
    "sales_Q3": [130, 210, 170]
})

df

,city,sales_Q1,sales_Q2,sales_Q3
0,Pune,100,120,130
1,Mumbai,200,220,210
2,Delhi,150,180,170


### `melt()` - Wide -> Long
`pd.melt()` **un-pivots** the DataFrame:   
- It keeps some columns fixed (identifier variables)
- It stacks multiple columns into **two columns:**
   - one for the column names
   - one for the values

#### Convert to tidy format

In [9]:
df_long = df.melt(
    id_vars = "city",
    value_vars = ["sales_Q1", "sales_Q2", "sales_Q3"],
    var_name = "quarter",
    value_name = "sales"
)

df_long

,city,quarter,sales
0,Pune,sales_Q1,100
1,Mumbai,sales_Q1,200
2,Delhi,sales_Q1,150
3,Pune,sales_Q2,120
4,Mumbai,sales_Q2,220
5,Delhi,sales_Q2,180
6,Pune,sales_Q3,130
7,Mumbai,sales_Q3,210
8,Delhi,sales_Q3,170


`id_vars="city"`
- Columns to **keep as identifiers**
- city will be repeated for each quarter

`value_vars=["sales_Q1", "sales_Q2", "sales_Q3"]`
- Columns to **unpivot / stack**

`var_name="quarter"`
- Name of the new column that will store old column names (`sales_Q1`, etc.)

`value_name="sales"`
- Name of the new column that will store the values

### `pivot()` - Long -> Wide (Strict)
Use when:
- Each (`index, column`) pair is **unique**
- No duplicates

`pivot()` **reshapes data** by:
- Turning **row values into columns**
- Using one column as the row index
- Filling cells with values from another column

It is essentially the **reverse of** `melt()`

In [10]:
df_pivot = df_long.pivot(
    index = "city",
    columns = "quarter",
    values = "sales"
)

df_pivot

quarter,sales_Q1,sales_Q2,sales_Q3
city,,,
Delhi,150,180,170
Mumbai,200,220,210
Pune,100,120,130


`index="city"`
- Rows will be grouped by **city**
- Each city becomes one row

`columns="quarter"`
- Unique values from `quarter` become **new columns**
- (`sales_Q1`, `sales_Q2`, `sales_Q3`)

`values="sales"`
- Cell values come from the `sales` column

### `pivot_table()` - Long -> Wide
Handles:
- Duplicate entries
- Aggregation

Use `pivot_table` in **production**      
`pivot` is strict and fragile

In [12]:
df_pivot_table = df_long.pivot_table(
    index = "city",
    columns = "quarter",
    values = "sales",
    aggfunc = "mean"
)

df_pivot_table

quarter,sales_Q1,sales_Q2,sales_Q3
city,,,
Delhi,150.0,180.0,170.0
Mumbai,200.0,220.0,210.0
Pune,100.0,120.0,130.0


### Resetting Index After Pivot

In [13]:
df_pivot_table.reset_index()

quarter,city,sales_Q1,sales_Q2,sales_Q3
0,Delhi,150.0,180.0,170.0
1,Mumbai,200.0,220.0,210.0
2,Pune,100.0,120.0,130.0


### `stack()` & `unstack()` (Index-Level Reshaping)

#### `stack()` - Columns -> Rows

In [15]:
df_stack = df.set_index("city").stack()
df_stack

city            
Pune    sales_Q1    100
        sales_Q2    120
        sales_Q3    130
Mumbai  sales_Q1    200
        sales_Q2    220
        sales_Q3    210
Delhi   sales_Q1    150
        sales_Q2    180
        sales_Q3    170
dtype: int64

#### `unstack()` - Rows -> Columns

In [17]:
df_unstack = df_stack.unstack()
df_unstack

,sales_Q1,sales_Q2,sales_Q3
city,,,
Pune,100,120,130
Mumbai,200,220,210
Delhi,150,180,170


`stack` / `unstack` are powerful but **less readable** than `melt` / `pivot`.

**User Guide:**
| Task                     | Use                     |
| ------------------------ | ----------------------- |
| Wide → Long              | `melt()`                |
| Long → Wide (unique)     | `pivot()`               |
| Long → Wide (duplicates) | `pivot_table()`         |
| Index-based reshape      | `stack()` / `unstack()` |

### ML Feature Engineering Example

#### Sales per quarter -> ML-ready format

In [19]:
df_long["quarter_num"] = df_long["quarter"].str.extract(r"Q(\d)").astype(int)
df_long

,city,quarter,sales,quarter_num
0,Pune,sales_Q1,100,1
1,Mumbai,sales_Q1,200,1
2,Delhi,sales_Q1,150,1
3,Pune,sales_Q2,120,2
4,Mumbai,sales_Q2,220,2
5,Delhi,sales_Q2,180,2
6,Pune,sales_Q3,130,3
7,Mumbai,sales_Q3,210,3
8,Delhi,sales_Q3,170,3


### Production Pattern

In [20]:
df_long = (
    df
    .melt(id_vars="city", var_name="quarter", value_name="sales")
    .assign(
        quarter_num = lambda x: x["quarter"].str.extract(r"Q(\d)").astype(int)
    )
)

### Exercise 1
```python
df = pd.DataFrame({
    "product": ["Laptop", "Tablet", "Mobile"],
    "sales_Q1": [500, 300, 700],
    "sales_Q2": [550, 320, 750],
    "sales_Q3": [600, 350, 800],
    "sales_Q4": [650, 400, 850]
})
```

1. Convert the DataFrame to **long format**
2. Columns should be:
   - `product`
   - `quarter`
   - `sales`

In [21]:
import pandas as pd

df = pd.DataFrame({
    "product": ["Laptop", "Tablet", "Mobile"],
    "sales_Q1": [500, 300, 700],
    "sales_Q2": [550, 320, 750],
    "sales_Q3": [600, 350, 800],
    "sales_Q4": [650, 400, 850]
})


In [25]:
df_long = df.melt(
    id_vars="product",
    value_vars=["sales_Q1", "sales_Q2", "sales_Q3", "sales_Q4"],
    value_name="sales",
    var_name="quarter"
)

df_long

,product,quarter,sales
0,Laptop,sales_Q1,500
1,Tablet,sales_Q1,300
2,Mobile,sales_Q1,700
3,Laptop,sales_Q2,550
4,Tablet,sales_Q2,320
5,Mobile,sales_Q2,750
6,Laptop,sales_Q3,600
7,Tablet,sales_Q3,350
8,Mobile,sales_Q3,800
9,Laptop,sales_Q4,650


### Exercise 2
From the `quarter` column:
1. Extract quarter number (`1`, `2`, `3`, `4`)
2. Store it in a new column `quarter_num`
3. Ensure dtype is integer

In [27]:
df_long["quarter_num"] = df_long["quarter"].str.extract(r"Q(\d)").astype(int)
df_long

,product,quarter,sales,quarter_num
0,Laptop,sales_Q1,500,1
1,Tablet,sales_Q1,300,1
2,Mobile,sales_Q1,700,1
3,Laptop,sales_Q2,550,2
4,Tablet,sales_Q2,320,2
5,Mobile,sales_Q2,750,2
6,Laptop,sales_Q3,600,3
7,Tablet,sales_Q3,350,3
8,Mobile,sales_Q3,800,3
9,Laptop,sales_Q4,650,4


### Exercise 3
1. Convert the long DataFrame back to **wide format**
2. Use:
   - index -> `product`
   - columns -> `quarter`
   - values -> `sales`

In [29]:
df_pivot = df_long.pivot(
    index="product",
    columns="quarter",
    values="sales"
)

df_pivot

quarter,sales_Q1,sales_Q2,sales_Q3,sales_Q4
product,,,,
Laptop,500,550,600,650
Mobile,700,750,800,850
Tablet,300,320,350,400


### Exercise 4
Assume duplicate data exists.
1. Use `pivot_table` to compute:
   - **average sales per product per quarter**
2. Reset the index to get a flat DataFrame

In [31]:
df_pivot_table = df_long.pivot_table(
    index="product",
    columns="quarter",
    values="sales",
    aggfunc="mean"
).reset_index()

df_pivot_table

quarter,product,sales_Q1,sales_Q2,sales_Q3,sales_Q4
0,Laptop,500.0,550.0,600.0,650.0
1,Mobile,700.0,750.0,800.0,850.0
2,Tablet,300.0,320.0,350.0,400.0


### Exercise 5
1. Set `product` as index
2. Apply `stack()`
3. Then apply `unstack()` and confirm it matches the original wide DataFrame

In [32]:
df_indexed = df.set_index("product")
df_indexed

,sales_Q1,sales_Q2,sales_Q3,sales_Q4
product,,,,
Laptop,500,550,600,650
Tablet,300,320,350,400
Mobile,700,750,800,850


In [34]:
df_stacked = df_indexed.stack()
df_stacked

product          
Laptop   sales_Q1    500
         sales_Q2    550
         sales_Q3    600
         sales_Q4    650
Tablet   sales_Q1    300
         sales_Q2    320
         sales_Q3    350
         sales_Q4    400
Mobile   sales_Q1    700
         sales_Q2    750
         sales_Q3    800
         sales_Q4    850
dtype: int64

In [35]:
df_unstacked = df_stacked.unstack()
df_unstacked

,sales_Q1,sales_Q2,sales_Q3,sales_Q4
product,,,,
Laptop,500,550,600,650
Tablet,300,320,350,400
Mobile,700,750,800,850
